# 06 — Forward odds

Collect **observed** two-sided quotes via ParlayAPI, apply frozen multiplicative no-vig,
and compare model p̂ to q_no-vig as a **model–market gap under mismatched estimands**.
CLV requires decision-time observed prices vs a valid closing benchmark — not p̂ alone.

Requires `PARLAY_API_KEY` in the repo-root `.env`. Kernel cwd may be `notebooks/`; paths resolve to the repo root.

In [14]:
from __future__ import annotations

import os
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd

from tml.features.elo import OVERALL_SURFACE, EloState, update_tournament
from tml.models.elo_prob import elo_win_prob
from tml.odds.client import fetch_h2h
from tml.odds.closing import select_closing_quote
from tml.odds.devig import multiplicative_novig
from tml.odds.storage import QuoteStore
from tml.shared.config import get_settings
from tml.shared.result import Err, Ok

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

# Settings / QuoteStore use paths relative to process cwd and `.env`.
os.chdir(REPO_ROOT)
get_settings.cache_clear()
settings = get_settings()

QUOTE_PATH = REPO_ROOT / "data" / "raw" / "odds" / "quotes.parquet"
store = QuoteStore(QUOTE_PATH)

key_ok = bool(settings.parlay_api_key and settings.parlay_api_key.strip())
print("repo", REPO_ROOT)
print("quote_path", QUOTE_PATH)
print("PARLAY_API_KEY set", key_ok)


repo /Users/alexgonzalez/Documents/tennis moneyline predictions
quote_path /Users/alexgonzalez/Documents/tennis moneyline predictions/data/raw/odds/quotes.parquet
PARLAY_API_KEY set True


In [15]:
# COLLECT=True hits ParlayAPI and appends; False reuses the parquet on disk.
COLLECT = not store.path.exists()
SPORT_KEY = "tennis_atp"

if COLLECT:
    result = fetch_h2h(SPORT_KEY)
    if isinstance(result, Err):
        raise RuntimeError(result.error)
    fetched = result.value
    if not fetched:
        raise RuntimeError(f"no h2h quotes returned for {SPORT_KEY}")
    store.append(fetched)
    print(f"appended {len(fetched)} quotes -> {store.path}")
elif not store.path.exists():
    raise FileNotFoundError(
        f"No quotes at {store.path}. Set COLLECT=True (needs PARLAY_API_KEY)."
    )

quotes = store.load()
print("stored quotes", len(quotes))


stored quotes 76


In [16]:
rows = []
for q in quotes:
    novig = multiplicative_novig(q.price_a, q.price_b, odds_format=q.odds_format)
    if isinstance(novig, Err):
        market_p_a = market_p_b = float("nan")
        novig_err = novig.error
    else:
        market_p_a, market_p_b = novig.value
        novig_err = None
    rows.append(
        {
            "event_id": q.event_id,
            "bookmaker": q.bookmaker,
            "player_a": q.player_a,
            "player_b": q.player_b,
            "price_a": q.price_a,
            "price_b": q.price_b,
            "market_p_a_wins": market_p_a,
            "market_p_b_wins": market_p_b,
            "novig_err": novig_err,
            "commence_time": q.commence_time,
            "last_update": q.last_update,
            "collected_at": q.collected_at,
            "is_delayed_reserve": q.is_delayed_reserve,
        }
    )

odds_df = pd.DataFrame(rows).sort_values(
    ["commence_time", "event_id", "bookmaker"], na_position="last"
)
display(
    odds_df[
        [
            "commence_time",
            "bookmaker",
            "player_a",
            "market_p_a_wins",
            "player_b",
            "market_p_b_wins",
        ]
    ].head(20)
)
print("events", odds_df["event_id"].nunique(), "books", odds_df["bookmaker"].nunique())
print("market_p_*_wins = multiplicative no-vig implied P(that player wins).")


,commence_time,bookmaker,player_a,market_p_a_wins,player_b,market_p_b_wins
18,2026-09-07 05:30:00+00:00,pinnacle,Dong Ju Kim,0.467870,Kristjan Tamm,0.532130
0,2026-09-07 06:30:00+00:00,pinnacle,Aditya Vishal Balsekar,0.194393,Hunter Heck,0.805607
45,2026-09-07 06:30:00+00:00,pinnacle,Martin Borisiouk,0.808386,Ethan Cook,0.191614
16,2026-09-07 07:00:00+00:00,pinnacle,Denis Yevseyev,0.681535,Hanyi Liu,0.318465
66,2026-09-07 07:00:00+00:00,pinnacle,Santiago Rodriguez Taverna,0.411914,Izan Almazan Valiente,0.588086
63,2026-09-07 07:00:00+00:00,pinnacle,Pierre Antoine Tailleu,0.373794,Mathieu Scaglia,0.626206
53,2026-09-07 07:00:00+00:00,pinnacle,Mika Brunold,0.625565,Matthew William Donald,0.374435
7,2026-09-07 07:00:00+00:00,pinnacle,Andrej Martin,0.424884,Svyatoslav Gulin,0.575116
12,2026-09-07 07:00:00+00:00,pinnacle,Benjamin Hassan,0.853401,Samir Hamza Reguig,0.146599
17,2026-09-07 07:00:00+00:00,pinnacle,Diego Dedura,0.532130,Henry Bernet,0.467870


events 75 books 2
market_p_*_wins = multiplicative no-vig implied P(that player wins).


In [17]:
# Closing selection: valid only when an observed quote sits inside the cutoff window.
# Early forward snapshots usually return Err (quote too old vs commence_time max_age).
event_id = odds_df["event_id"].iloc[0]
event_quotes = [q for q in quotes if q.event_id == event_id]
commence = event_quotes[0].commence_time

if commence is None:
    print("event has no commence_time; skipping closing demo")
else:
    closing = select_closing_quote(event_quotes, cutoff=commence)
    if isinstance(closing, Ok):
        c = closing.value
        print(
            "closing",
            c.bookmaker,
            c.player_a,
            c.price_a,
            c.player_b,
            c.price_b,
            c.last_update,
        )
    else:
        print("closing:", closing.error)
        print(
            "(expected for early snapshots — re-collect near start for a valid closer)"
        )

    # Illustrative: decision-time cutoff = now (still needs a quote within max_age).
    now = datetime.now(timezone.utc)
    decision = select_closing_quote(
        event_quotes,
        cutoff=now,
        buffer=timedelta(minutes=0),
        max_age=timedelta(hours=6),
    )
    print(
        "decision-window quote:",
        decision.value.bookmaker if isinstance(decision, Ok) else decision.error,
    )

closing: no valid observed closing quote
(expected for early snapshots — re-collect near start for a valid closer)
decision-window quote: pinnacle


In [18]:
# Model–market gap under mismatched estimands (not CLV).
# model_p_*_wins = B1 Overall Elo P(that player wins | match completes).
# market_p_*_wins = multiplicative no-vig from observed prices.

import re
import unicodedata


def _normalize_name(value: str) -> str:
    ascii_value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode()
    return " ".join(re.findall(r"[a-z0-9]+", ascii_value.casefold()))


def _name_to_player_id(data_dir: Path) -> dict[str, str]:
    """Map normalized Sackmann names -> unique player_id (skip collisions)."""
    buckets: dict[str, set[str]] = {}
    files = [
        path
        for path in sorted(data_dir.glob("20*.csv"))
        if "wta" not in path.name.casefold()
        and "ranking" not in path.name.casefold()
    ]
    for path in files:
        try:
            frame = pd.read_csv(
                path,
                usecols=lambda c: c
                in {"winner_id", "loser_id", "winner_name", "loser_name"},
            )
        except (ValueError, OSError):
            continue
        for side in ("winner", "loser"):
            for player_id, name in zip(
                frame[f"{side}_id"].astype(str),
                frame[f"{side}_name"].astype(str),
                strict=True,
            ):
                buckets.setdefault(_normalize_name(name), set()).add(player_id)
    return {
        name: next(iter(ids)) for name, ids in buckets.items() if len(ids) == 1
    }


data_dir = Path(settings.tml_data_dir)
if not data_dir.exists():
    data_dir = REPO_ROOT / "notebooks" / "tml-data"
name_to_id = _name_to_player_id(data_dir)

matches = pd.read_parquet(REPO_ROOT / "data" / "processed" / "modeling.parquet")
matches["tourney_date"] = pd.to_datetime(matches["tourney_date"])
history = matches.loc[matches["tourney_date"] >= "2018-01-01"].copy()
state = EloState()
for _, tournament in history.groupby(["tourney_date", "tourney_id"], sort=True):
    update_tournament(state, tournament)


def _model_p_a_wins(player_a: str, player_b: str) -> float | None:
    id_a = name_to_id.get(_normalize_name(player_a))
    id_b = name_to_id.get(_normalize_name(player_b))
    if id_a is None or id_b is None:
        return None
    return float(
        elo_win_prob(state, id_a, id_b, OVERALL_SURFACE, best_of=3)
    )


gap_df = odds_df.dropna(subset=["market_p_a_wins"]).copy()
gap_df["model_p_a_wins"] = [
    _model_p_a_wins(a, b)
    for a, b in zip(gap_df["player_a"], gap_df["player_b"], strict=True)
]
gap_df["model_p_b_wins"] = 1.0 - gap_df["model_p_a_wins"]
gap_df["gap_a_model_minus_market"] = (
    gap_df["model_p_a_wins"] - gap_df["market_p_a_wins"]
)

n_scored = gap_df["model_p_a_wins"].notna().sum()
display(
    gap_df.loc[gap_df["model_p_a_wins"].notna()]
    .sort_values(
        "gap_a_model_minus_market", key=lambda s: s.abs(), ascending=False
    )[
        [
            "bookmaker",
            "player_a",
            "model_p_a_wins",
            "market_p_a_wins",
            "player_b",
            "model_p_b_wins",
            "market_p_b_wins",
            "gap_a_model_minus_market",
        ]
    ]
    .head(15)
)
print(
    f"scored {n_scored}/{len(gap_df)} quote rows "
    f"({data_dir}; {len(name_to_id)} unique name→id)"
)
print("Unmatched names leave model_p_*_wins as NA.")
print(
    "gap_a_model_minus_market > 0 means the model likes player_a more than the market."
)


,bookmaker,player_a,model_p_a_wins,market_p_a_wins,player_b,model_p_b_wins,market_p_b_wins,gap_a_model_minus_market
61,pinnacle,Omar Jasika,0.374282,0.790219,Yaojie Zeng,0.625718,0.209781,-0.415937
45,pinnacle,Martin Borisiouk,0.497182,0.808386,Ethan Cook,0.502818,0.191614,-0.311204
16,pinnacle,Denis Yevseyev,0.410433,0.681535,Hanyi Liu,0.589567,0.318465,-0.271102
0,pinnacle,Aditya Vishal Balsekar,0.457636,0.194393,Hunter Heck,0.542364,0.805607,0.263243
70,pinnacle,Timofei Derepasko,0.516173,0.765052,Taiyo Yamanaka,0.483827,0.234948,-0.248879
34,pinnacle,Ivan Marrero Curbelo,0.463413,0.227660,Maks Kasnikowski,0.536587,0.772340,0.235754
29,pinnacle,Gijs Brouwer,0.578889,0.365743,Matteo Martineau,0.421111,0.634257,0.213146
5,pinnacle,Alexandr Binda,0.444753,0.655203,Mikalai Haliak,0.555247,0.344797,-0.210450
39,pinnacle,Linang Xiao,0.558800,0.357995,Kaichi Uchida,0.441200,0.642005,0.200805
8,pinnacle,Andrey Chepelev,0.453183,0.632572,Gianmarco Ferrari,0.546817,0.367428,-0.179389


scored 56/76 quote rows (/Users/alexgonzalez/Documents/tennis moneyline predictions/notebooks/tml-data; 5549 unique name→id)
Unmatched names leave model_p_*_wins as NA.
gap_a_model_minus_market > 0 means the model likes player_a more than the market.
